<a href="https://colab.research.google.com/github/denis20vv2/DeepNeuro/blob/main/Lab8%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Работа №8**

In [ ]:


!pip install transformers --quiet

from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import files
import torch
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

with open(file_path, "r", encoding="utf-8") as f:
    data = f.read().replace("\n", " ")

model_name = "Qwen/Qwen2.5-7B-Instruct-1M"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

questions = [
    "В каком году была обозначена проблема взрывающихся градиентов?",
    "Кто в 1891 году разработал метод уничтожающей производной?",
    "Кто предложил цепное правило дифференцирования и в каком году?"
]

max_tokens = 2000
tokens = tokenizer(data)["input_ids"]
chunks = [tokens[i:i+max_tokens] for i in range(0, len(tokens), max_tokens)]

def ask_llm(chunk_ids, questions):
    prompt_text = tokenizer.decode(chunk_ids, skip_special_tokens=True)
    prompt_text += "\n\nПрочитай текст выше и ответь строго на вопросы.\n"
    for i, q in enumerate(questions, 1):
        prompt_text += f"{i}. {q}\n"
    prompt_text += "\nОтвет:\n"

    inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    lines = [line.strip() for line in response.split("\n") if line.strip()]
    return "\n".join(lines)

all_answers = []

for i, chunk in enumerate(chunks, 1):
    print(f"Обрабатываем кусок {i}/{len(chunks)}...")
    response = ask_llm(chunk, questions)
    all_answers.append(response)

print("\n======= ОТВЕТЫ ОТ LLM =======\n")
for ans in all_answers:
    print(ans)
    print("-"*50)


Saving ENG_article.txt to ENG_article (37).txt


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Обрабатываем кусок 1/2...
